In [2]:
# Compare two MLP architectures on the Heart Disease dataset.
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


In [3]:

# Remove repeated records before splitting to prevent identical rows crossing the split.
df = pd.read_csv("heart.csv")
print("Rows before deduplication:", len(df))
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after deduplication:", len(df))
print("Target counts:\n", df["target"].value_counts().sort_index())
print("Missing values:", int(df.isna().sum().sum()))


Rows before deduplication: 1025
Rows after deduplication: 302
Target counts:
 target
0    138
1    164
Name: count, dtype: int64
Missing values: 0


In [4]:

# Use every column except target as input; stratification preserves class proportions.
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:

# Scaling is important for MLPs; pipelines ensure scaling is fit on training data only.
architectures = {
    "MLP (32, 16)": (32, 16),
    "MLP (64, 32, 16)": (64, 32, 16),
}
results = []


In [6]:

# Keep the split and training settings identical for a fair architecture comparison.
for name, hidden_layers in architectures.items():
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=hidden_layers, max_iter=1000, random_state=42),
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({
        "Architecture": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
    })


In [7]:

# Rank by F1, then accuracy, to favor balanced performance with a clear tie-breaker.
comparison = pd.DataFrame(results).sort_values(
    ["F1 Score", "Accuracy"], ascending=False
).reset_index(drop=True)
display(comparison.round(3))

best = comparison.iloc[0]
print(
    f"Better architecture: {best['Architecture']} (selected by F1 Score, then Accuracy). "
    f"Accuracy={best['Accuracy']:.3f}, Precision={best['Precision']:.3f}, "
    f"Recall={best['Recall']:.3f}, F1={best['F1 Score']:.3f}. "
    "It is preferred because it has the strongest balanced test-set performance "
    "under this split; the table shows the trade-offs across all four metrics."
)

,Architecture,Accuracy,Precision,Recall,F1 Score
0,"MLP (32, 16)",0.754,0.765,0.788,0.776
1,"MLP (64, 32, 16)",0.754,0.781,0.758,0.769


Better architecture: MLP (32, 16) (selected by F1 Score, then Accuracy). Accuracy=0.754, Precision=0.765, Recall=0.788, F1=0.776. It is preferred because it has the strongest balanced test-set performance under this split; the table shows the trade-offs across all four metrics.
